# 📊 Trader Behavior Insights: Market Sentiment vs Performance

This **Google Colab** notebook analyzes the relationship between Bitcoin market sentiment (Fear vs Greed) and trader behavior using Hyperliquid historical data.
It implements feature engineering like the **Risk Proxy** and performs **Trader Clustering**.

In [ ]:
# 0. GOOGLE COLAB DATA PIPELINE
# Download the datasets directly from Google Drive
!pip install -q gdown
!gdown --id 1IAfLZwu6rJzyWKgBToqwSmmVYU6VbjVs -O historical_data.csv
!gdown --id 1PgQC0tO8XN-wqkNyghWc_-mnrYv_nhSf -O fear_greed_index.csv

print('\nDatasets downloaded successfully into the Colab environment!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Elite-tier visualization styling
plt.style.use('dark_background')
sns.set_palette('muted')
sns.set_context('talk')

In [ ]:
# 1. DATA PREPROCESSING
print('Loading datasets...')
try:
    df_trades = pd.read_csv('historical_data.csv')
    df_sentiment = pd.read_csv('fear_greed_index.csv')
    print('Data loaded successfully.')
except FileNotFoundError:
    print('Error: Data failed to load. Please re-run the dataset download block.')

df_trades.columns = [c.strip() for c in df_trades.columns]
df_sentiment.columns = [c.strip() for c in df_sentiment.columns]

if 'Date' in df_sentiment.columns:
    df_sentiment.rename(columns={'Date': 'date'}, inplace=True)
if 'Classification' in df_sentiment.columns:
    df_sentiment.rename(columns={'Classification': 'sentiment'}, inplace=True)

df_sentiment['date'] = pd.to_datetime(df_sentiment['date']).dt.date

# Robust Time Parsing (handling milliseconds properly)
time_col = next((c for c in df_trades.columns if c.lower() in ['time', 'timestamp', 'date', 'datetime']), None)
if time_col:
    if str(df_trades[time_col].iloc[0]).replace('.', '').isdigit() and len(str(df_trades[time_col].iloc[0])) >= 12:
        df_trades['time'] = pd.to_datetime(pd.to_numeric(df_trades[time_col]), unit='ms')
    else:
        df_trades['time'] = pd.to_datetime(df_trades[time_col], errors='coerce')
    df_trades.dropna(subset=['time'], inplace=True)
    df_trades['date'] = df_trades['time'].dt.date
else:
    print('Warning: No recognizable time logic found.')

df_sentiment['sentiment'] = df_sentiment['sentiment'].replace({'Extreme Fear': 'Fear', 'Extreme Greed': 'Greed'})
df_merged = pd.merge(df_trades, df_sentiment, on='date', how='inner')
print(f'Merged Data Shape: {df_merged.shape}')

In [ ]:
# 2. FEATURE ENGINEERING (ROBUST VERSION)
df_merged.columns = [c.lower() for c in df_merged.columns]

# 1. Assign Profit 
pnl_col = next((c for c in df_merged.columns if 'pnl' in c), None)
if pnl_col:
    df_merged['profit'] = pd.to_numeric(df_merged[pnl_col], errors='coerce').fillna(0)
else:
    df_merged['profit'] = 0.0

df_merged['win'] = (df_merged['profit'] > 0).astype(int)

# 2. Extract Size
if 'size' in df_merged.columns:
    df_merged['numeric_size'] = pd.to_numeric(df_merged['size'], errors='coerce').fillna(1)
elif 'size usd' in df_merged.columns:
    df_merged['numeric_size'] = pd.to_numeric(df_merged['size usd'], errors='coerce').fillna(1)
else:
    df_merged['numeric_size'] = 1000

# 3. Dynamic Trade Frequency Locator
account_col = next((c for c in df_merged.columns if c in ['account', 'user', 'trader', 'id']), None)

if account_col:
    df_merged.rename(columns={account_col: 'account'}, inplace=True)
    trade_freq = df_merged.groupby(['account', 'date']).size().reset_index(name='trade_frequency')
    df_merged = pd.merge(df_merged, trade_freq, on=['account', 'date'], how='left')
else:
    df_merged['account'] = 'Aggregated_Trader'
    trade_freq = df_merged.groupby('date').size().reset_index(name='trade_frequency')
    df_merged = pd.merge(df_merged, trade_freq, on=['date'], how='left')

if 'trade_frequency' not in df_merged.columns:
    df_merged['trade_frequency'] = 1

# 4. --- HERO METRIC: RISK PROXY ---
df_merged['risk_proxy'] = df_merged['numeric_size'] * df_merged['trade_frequency']

# 5. Sentiment Shift Feature
df_sentiment_sorted = df_sentiment.sort_values('date').copy()
df_sentiment_sorted.columns = [c.lower() for c in df_sentiment_sorted.columns]
df_sentiment_sorted['prev_sentiment'] = df_sentiment_sorted['sentiment'].shift(1)
df_merged = pd.merge(df_merged, df_sentiment_sorted[['date', 'prev_sentiment']], on='date', how='left')
df_merged['sentiment_shift'] = df_merged['prev_sentiment'].astype(str) + ' \u2192 ' + df_merged['sentiment'].astype(str)

print('Feature Engineering Complete. Engineered metrics: profit, win_rate, trade_frequency, numeric_size, risk_proxy')

In [ ]:
# 3. CORE ANALYSIS: Profitability, Risk, and Activity vs Sentiment
print('\n--- CORE ANALYSIS BY SENTIMENT ---')
sentiment_stats = df_merged.groupby('sentiment').agg(
    avg_profit=('profit', 'mean'),
    win_rate=('win', 'mean'),
    avg_trade_freq=('trade_frequency', 'mean'),
    avg_size=('numeric_size', 'mean'),
    avg_risk_proxy=('risk_proxy', 'mean')
).reset_index()
display(sentiment_stats)

In [ ]:
# 4. ADVANCED ANALYSIS: Sentiment Shift & Trader Profiling
print('\n--- SENTIMENT SHIFT IMPACT: Fear \u2192 Greed ---')
try:
    shift_stats = df_merged[df_merged['sentiment_shift'] == 'Fear \u2192 Greed'].agg(
        avg_profit=('profit', 'mean'),
        win_rate=('win', 'mean'),
        avg_risk_proxy=('risk_proxy', 'mean')
    )
    print('Performance deeply shifting from Fear \u2192 Greed:')
    display(shift_stats)
except:
    print('No Fear \u2192 Greed specific transitions found in this timeframe slice.')

print('\n--- SMART MONEY VS WEAK TRADERS ---')
account_profits = df_merged.groupby('account')['profit'].sum().sort_values(ascending=False)
top_20 = account_profits.quantile(0.8)
bottom_20 = account_profits.quantile(0.2)

df_merged['trader_type'] = 'Average'
df_merged.loc[df_merged['account'].isin(account_profits[account_profits >= top_20].index), 'trader_type'] = 'Smart Money'
df_merged.loc[df_merged['account'].isin(account_profits[account_profits <= bottom_20].index), 'trader_type'] = 'Weak Traders'

trader_comp = df_merged[df_merged['trader_type'] != 'Average'].groupby(['trader_type', 'sentiment'])[['risk_proxy', 'win', 'trade_frequency']].mean().reset_index()
display(trader_comp)

print('\n--- TRADER CLUSTERING (BEHAVIOR PROFILING) ---')
trader_features = df_merged.groupby('account').agg(
    avg_profit=('profit', 'mean'), 
    avg_risk_proxy=('risk_proxy', 'mean'), 
    win_rate=('win', 'mean'), 
    total_size=('numeric_size', 'sum')
).fillna(0)

kmeans = KMeans(n_clusters=3, random_state=42)
trader_features['cluster'] = kmeans.fit_predict(trader_features)
print('\nCluster Centroids (Behavior Groups):')
display(trader_features.groupby('cluster').mean())

In [ ]:
# 5. VISUALIZATIONS FOR INTELLIGENCE REPORT
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Quantitative Behavioral Impact of Market Sentiment', fontsize=24, color='#00d4ff', fontweight='bold')

# 1. Profit vs Sentiment
q_low = df_merged['profit'].quantile(0.05)
q_high = df_merged['profit'].quantile(0.95)
filtered_df = df_merged[(df_merged['profit'] > q_low) & (df_merged['profit'] < q_high)]
sns.boxplot(data=filtered_df, x='sentiment', y='profit', ax=axes[0,0], showfliers=False, palette='coolwarm')
axes[0,0].set_title('Profitability per Sentiment Phase', fontsize=18)
axes[0,0].axhline(0, color='grey', linestyle='--', alpha=0.5)

# 2. Win Rate vs Sentiment
sns.barplot(data=sentiment_stats, x='sentiment', y='win_rate', ax=axes[0,1], palette='magma')
axes[0,1].set_title('Average Win Rate by Sentiment', fontsize=18)

# 3. Risk Proxy Use: Smart vs Weak
sns.violinplot(data=df_merged, x='sentiment', y='risk_proxy', hue='trader_type', split=True, ax=axes[1,0], palette='viridis', cut=0)
axes[1,0].set_title('Risk Proxy Discipline: Smart Money vs Weak Traders', fontsize=18)
axes[1,0].set_yscale('log')

# 4. Risk Proxy vs Sentiment
sns.barplot(data=sentiment_stats, x='sentiment', y='avg_risk_proxy', ax=axes[1,1], palette='flare')
axes[1,1].set_title('Risk Proxy & Exposure by Sentiment Phase', fontsize=18)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('sentiment_behavior_insights.png', dpi=300, facecolor='black')
plt.show()

# Cluster Visualization
plt.figure(figsize=(12,7))
sns.scatterplot(data=trader_features, x='avg_risk_proxy', y='win_rate', hue='cluster', size='total_size', sizes=(50, 600), palette='deep', alpha=0.8)
plt.title('Trader Behavior Clustering (Risk Proxy vs Win Rate)', fontsize=20, color='#00d4ff')
plt.xlabel('Average Risk Proxy')
plt.ylabel('Win Rate')
plt.xscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.savefig('trader_clusters.png', dpi=300, facecolor='black')
plt.show()

In [ ]:
# 6. PREDICTIVE MODEL (Machine Learning)
print('\n--- PREDICTIVE MODEL ---')
model_features = ['trade_frequency', 'numeric_size', 'risk_proxy']
df_ml = df_merged.dropna(subset=model_features + ['win']).copy()
X = df_ml[model_features].copy()
if 'sentiment' in df_ml.columns:
    X['is_greed'] = (df_ml['sentiment'] == 'Greed').astype(int)
y = df_ml['win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print('Random Forest Accuracy (Predicting Trade Win):', round(accuracy_score(y_test, y_pred), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

feat_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(8,4))
feat_importances.plot(kind='barh', color='teal')
plt.title('Feature Importances for Profitability Prediction', color='#00d4ff')
plt.show()
